# Contoso Forge: interactive Spark → BigQuery
Experimental generated notebook; cloud execution has not been validated. Use a small CPU runtime.
Before packaging, set the GCP project/dataset/location in project.json and regenerate. Create the selected dataset manually in BigQuery Sandbox (no GCS required), or review the generated infrastructure for a billing-enabled project. Authenticate as a user with dataset write and job creation access.
On your computer, run `python colab/work_order.py package --root . --run-id demo-001`. Upload its work_package.zip below. The notebook reuses the V1 CDC/SCD2/quality rules, writes Parquet, loads native BigQuery tables, queries actual counts/KPIs, and returns a result manifest. Airflow cannot start this notebook unattended.
Each work order uses unique table names and WRITE_EMPTY loads. Sandbox tables expire according to Sandbox policy. Billing-enabled free allowances are not a guarantee of zero charges; query bytes and file size are capped by generated config.

In [ ]:
%pip install -q pyspark==3.5.9 pyarrow==19.0.1 google-cloud-bigquery==3.44.0
import os, sys, subprocess, json, io, zipfile, stat, uuid
from pathlib import Path
# Colab provides Java; fail visibly if the selected runtime does not.
subprocess.run(['java', '-version'], check=True)

In [ ]:
from google.colab import files
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload exactly one work_package.zip'
root = Path('/content') / ('contoso_' + uuid.uuid4().hex)
root.mkdir()
with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as package:
    names = set()
    assert sum(x.file_size for x in package.infolist()) <= 500_000_000, 'Package exceeds the small-lab limit'
    for item in package.infolist():
        target = (root / item.filename).resolve()
        assert target.is_relative_to(root) and target != root and '\\' not in item.filename, 'Unsafe ZIP path'
        assert item.filename not in names and not stat.S_ISLNK(item.external_attr >> 16), 'Unsafe ZIP member'
        names.add(item.filename)
    package.extractall(root)
os.chdir(root)
sys.path.insert(0, str(root / 'colab'))
from work_order import read_json, validate_order
order = read_json(root / 'colab/work_order.json')
validate_order(root, order)
print('Work order:', order['workOrderId'], 'Run:', order['runId'])
print('Dataset:', order['gcp']['projectId'] + '.' + order['gcp']['dataset'])

In [ ]:
subprocess.run([sys.executable, 'colab/run_spark.py', '--root', '.', '--lake-root', 'lake', '--work-order', 'colab/work_order.json'], check=True)

In [ ]:
from google.colab import auth
auth.authenticate_user()
# Uses the signed-in user's standard credentials. No credential is stored in the package.

In [ ]:
subprocess.run([sys.executable, 'gcp/bigquery_runtime.py', 'run', '--root', '.', '--silver-root', 'lake/silver', '--work-order', 'colab/work_order.json', '--result', 'colab/result_manifest.json'], check=True)

In [ ]:
files.download(str(root / 'colab/result_manifest.json'))
print('Return this file to the matching Airflow run state directory, or reconcile locally with:')
print('python colab/work_order.py reconcile --root . --work-order colab/work_order.json --result colab/result_manifest.json')